In [7]:
!pip install numpy pandas scikit-learn matplotlib tqdm
!pip install git+https://github.com/moment-timeseries-foundation-model/moment.git

  Cloning https://github.com/moment-timeseries-foundation-model/moment.git to /private/var/folders/rs/xp_jvmcs6yxft09g6r_s670m0000gn/T/pip-req-build-e8_4k053
  Running command git clone --filter=blob:none --quiet https://github.com/moment-timeseries-foundation-model/moment.git /private/var/folders/rs/xp_jvmcs6yxft09g6r_s670m0000gn/T/pip-req-build-e8_4k053
  Resolved https://github.com/moment-timeseries-foundation-model/moment.git to commit 38f7310ad594100747ca2a8357e9c7ca7d323e0e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [8]:
from momentfm import MOMENTPipeline
from pprint import pprint
import torch

In [9]:
model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large", 
    model_kwargs={"task_name": "reconstruction"},  # For anomaly detection load MOMENT reconstruction mode
)

In [10]:
model.init()
print(model)

MOMENTPipeline(
  (normalizer): RevIN()
  (tokenizer): Patching()
  (patch_embedding): PatchEmbedding(
    (value_embedding): Linear(in_features=8, out_features=1024, bias=False)
    (position_embedding): PositionalEmbedding()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
  

In [11]:
# look at weights
# names, shapes, and which ones are trainable
for name, param in model.named_parameters():
    print(name, tuple(param.shape), param.requires_grad)

# the full weight dictionary
sd = model.state_dict()
sd.keys()                      # every parameter/buffer name
sd["encoder.block.0.layer.0.SelfAttention.q.weight"]   # one specific tensor

# actual values, as numpy
w = sd["encoder.block.0.layer.0.SelfAttention.q.weight"].detach().cpu().numpy()

patch_embedding.mask_embedding (1024,) False
patch_embedding.value_embedding.weight (1024, 8) False
encoder.embed_tokens.weight (32128, 1024) False
encoder.block.0.layer.0.SelfAttention.q.weight (1024, 1024) False
encoder.block.0.layer.0.SelfAttention.k.weight (1024, 1024) False
encoder.block.0.layer.0.SelfAttention.v.weight (1024, 1024) False
encoder.block.0.layer.0.SelfAttention.o.weight (1024, 1024) False
encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight (32, 16) False
encoder.block.0.layer.0.layer_norm.weight (1024,) False
encoder.block.0.layer.1.DenseReluDense.wi_0.weight (2816, 1024) False
encoder.block.0.layer.1.DenseReluDense.wi_1.weight (2816, 1024) False
encoder.block.0.layer.1.DenseReluDense.wo.weight (1024, 2816) False
encoder.block.0.layer.1.layer_norm.weight (1024,) False
encoder.block.1.layer.0.SelfAttention.q.weight (1024, 1024) False
encoder.block.1.layer.0.SelfAttention.k.weight (1024, 1024) False
encoder.block.1.layer.0.SelfAttention.v.weight (1024

In [12]:
# load data

from pathlib import Path
from data_loading import list_condition_stems, load_condition

data_dir = Path("Data")  # contains vibration/ and current,temp/

# Every condition that has BOTH files on disk (warns on any that don't)
stems = list_condition_stems(data_dir)
print(f"{len(stems)} paired conditions:", stems[:5], "...")

# Load one condition
rec = load_condition(data_dir, "0Nm_Normal")

# print(rec.info)                              # ConditionInfo(load_nm=2, condition='BPFO', severity='03', ...)
# print(rec.vibration.values)                  # (n_samples, 4) float64, units g
# print(rec.vibration.sample_rate_hz)          # 25600.0
# print(rec.vibration.channel_names)           # ['accel_x_A', 'accel_y_A', 'accel_x_B', 'accel_y_B']

print(rec.current_temp.temperature)          # (n_samples, 2) float64, °C
print(rec.current_temp.current)              # (n_samples, k) float64, A -- k varies by file!
print(rec.current_temp.current_channels)     # which phase(s) actually had data
print(rec.current_temp.sample_rate_hz)       # ~25608 Hz (slightly different from vibration's rate)

[data] WARNING: no .tdms match for: ['2Nm_Unbalalnce_0583mg', '2Nm_Unbalalnce_1169mg', '2Nm_Unbalalnce_1751mg', '2Nm_Unbalalnce_2239mg', '2Nm_Unbalalnce_3318mg']
[data] WARNING: no .mat match for: ['2Nm_Unbalance_0583mg', '2Nm_Unbalance_1169mg', '2Nm_Unbalance_1751mg', '2Nm_Unbalance_2239mg', '2Nm_Unbalance_3318mg']
40 paired conditions: ['0Nm_BPFI_03', '0Nm_BPFI_10', '0Nm_BPFI_30', '0Nm_BPFO_03', '0Nm_BPFO_10'] ...
[[24.95391806 25.1877301 ]
 [24.95391806 25.1877301 ]
 [24.95391806 25.1877301 ]
 ...
 [26.991856   27.23889551]
 [26.991856   27.23889551]
 [26.991856   27.23889551]]
[[ 1.85872054  0.79149249 -2.08180909]
 [ 2.11791797  0.78462421 -2.25151439]
 [ 2.06168996  0.93710002 -2.32106574]
 ...
 [ 3.25344959 -2.30060717 -0.6601795 ]
 [ 3.19996441 -2.4214889  -0.54611529]
 [ 2.96819527 -2.38577384 -0.34719843]]
['cDAQ9185-1F486B5Mod2/ai0', 'cDAQ9185-1F486B5Mod2/ai2', 'cDAQ9185-1F486B5Mod2/ai3']
25608.19462227913
